# 06. 추론 & 리소스 정리

> 이 노트북은 **개방형**입니다. 앞에서 익힌 패턴을 응용해 직접 코드를 작성해 보세요.
> 힌트와 문서 링크만 제공됩니다. (막히면 `solutions/` 참고)

## 이 노트북에서 배우는 것
- 배포된 엔드포인트를 **SageMaker SDK Predictor** 로 호출하는 헬퍼 함수 작성
- **boto3 `sagemaker-runtime`** 로 직접 호출 (웹앱이 사용하는 저수준 방식)
- 사용한 **리소스 정리(비용 절감)**

`05_deployment.ipynb` 를 먼저 실행해 엔드포인트가 떠 있어야 합니다.

## 0. 환경 복원 & Predictor 재생성

In [ ]:
import sagemaker
from sagemaker.predictor import Predictor
from sagemaker.serializers import CSVSerializer
from sagemaker.deserializers import CSVDeserializer
from sagemaker.s3 import S3Downloader
import pandas as pd

sess = sagemaker.Session()
%store -r endpoint_name region test_x_s3

predictor = Predictor(
    endpoint_name=endpoint_name,
    sagemaker_session=sess,
    serializer=CSVSerializer(),
    deserializer=CSVDeserializer(),
)

S3Downloader.download(test_x_s3, "infer")
X = pd.read_csv("infer/test_x.csv", header=None)
sample = X.iloc[0].tolist()
print("endpoint:", endpoint_name)
print("sample features:", sample[:6], "...")

## 1. 실시간 추론 헬퍼 함수 (직접 작성)

아래 `predict_student(features)` 를 완성하세요. 입력은 피처 리스트, 출력은 `(예측_라벨, 확률_딕셔너리)` 입니다.

**단계**
1. `predictor.predict(features)` 로 응답을 받습니다. 응답은 `[['p0','p1','p2']]` 형태입니다.
2. `resp[0]` 을 `float` 리스트로 변환합니다.
3. `np.argmax` 로 가장 큰 확률의 인덱스를 구해 `classes[idx]` 를 라벨로 만듭니다.

참고: https://sagemaker.readthedocs.io/en/v2.219.0/api/inference/predictors.html

In [ ]:
import numpy as np
classes = ["Dropout", "Enrolled", "Graduate"]

def predict_student(features):
    # TODO: 위 3단계를 구현하세요.
    ____

# 검증
label, proba = predict_student(sample)
print("prediction:", label)
print("proba     :", proba)

## 2. boto3 로 직접 호출 (저수준 방식)

SDK 없이 `sagemaker-runtime` 클라이언트로 호출하는 방법입니다. **웹 애플리케이션(07)** 이 이 방식을 씁니다.
`ContentType` 과 `Body`(CSV 문자열)를 채워 완성하세요.

참고: https://boto3.amazonaws.com/v1/documentation/api/latest/reference/services/sagemaker-runtime.html

In [ ]:
import boto3

runtime = boto3.client("sagemaker-runtime", region_name=region)
csv_row = ",".join(str(x) for x in sample)

# TODO: invoke_endpoint 호출을 완성하세요.
#  - EndpointName=endpoint_name
#  - ContentType="text/csv"
#  - Body=csv_row
resp = runtime.invoke_endpoint(
    EndpointName=endpoint_name,
    ContentType=____,
    Body=____,
)
print(resp["Body"].read().decode())

## 3. 🧹 리소스 정리 (반드시 실행!)

실시간 엔드포인트는 삭제 전까지 과금됩니다. **선택 과제(07 웹앱)를 할 예정이라면 이 셀은 웹앱 실습 이후에 실행**하세요.
그렇지 않다면 지금 바로 실행해 엔드포인트/모델을 삭제합니다.

In [ ]:
# 엔드포인트 + 엔드포인트 구성 + 모델 삭제
try:
    predictor.delete_model()
except Exception as e:
    print("delete_model:", e)
predictor.delete_endpoint()   # 엔드포인트와 엔드포인트 구성까지 삭제
print("cleaned up:", endpoint_name)

✅ **핵심 워크플로우 완주!** 전처리 → 학습 → 평가 → 튜닝 → 배포 → 추론까지 마쳤습니다.

### (선택) 다음 단계
`07_web_app.ipynb` — 이 엔드포인트를 호출하는 **웹 애플리케이션**을 만들어 로컬 개발 서버로 띄워 봅니다.
웹앱을 실습하려면 위 정리 셀을 아직 실행하지 마세요(엔드포인트가 필요합니다).